# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Thar Htet Zaw
* Student ID: 2503051A


**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

**Use of Generative AI**

Claude Code was used for project scaffolding, code review and structuring. All modelling decisions, metric justification and analysis are my own and I can explain each of them. _(Declaring this does not affect marks — see spec, Guidelines on using Gen AI.)_


# Libraries

In [ ]:
## Import libraries
import json
import time
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from scipy.io import arff
from scipy.stats import spearmanr

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import (
    train_test_split, cross_validate, cross_val_predict, StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)

# Fixed seed everywhere: the rubric's tuning row requires a reproducible setup.
RANDOM_STATE = 42

ARTIFACTS = Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


# 1. Business Understanding

**Goal:** flag phishing websites from 30 structural signals extracted from a URL and its page, so a browser extension / email gateway can warn a user *before* they enter credentials.

**Task type:** binary classification. Target `Result` — `-1` phishing, `1` legitimate.

**Cost of the two errors (this drives the metric choice in section 5):**

| Error | What happens | Who pays |
| --- | --- | --- |
| False negative — phishing scored legitimate | user is not warned, enters credentials | account takeover, financial loss |
| False positive — legitimate scored phishing | user sees a warning on a safe site | one dismissed banner |

There are two customers here, and they are not the same person. The end user is whoever is about to type a password into a page they arrived at from an email or a search result. That person is non-technical, in a hurry, and has no realistic way of telling a cloned login page from the genuine one. The deploying organisation, meaning an IT security team shipping this as a browser extension or embedding it in an email gateway, is the party that picks the operating point and then fields the complaints. PhishGuard sits between the two: it scores a page using 30 structural signals and decides whether to interrupt the user before the credential field rather than after it.

The two errors cost very different things.

A false negative is silent. Nobody warns the user, credentials go into an attacker-controlled page, and the outcome is account takeover, quite possibly followed by password reuse elsewhere or lateral movement if the account is a corporate one. The cost scales with whatever sits behind the account, so it has no fixed ceiling. It is also irreversible, since the credential is gone the instant it is submitted, and it falls entirely on the user, who never finds out that a model made a call at all.

A false positive is loud. The user sees a warning on a safe site, judges it wrong, and clicks through. One interruption is the whole cost, the site remains reachable, and the burden lands on the deploying organisation as a support ticket.

False positives still are not free, and that is what keeps the ratio from running away. A model that warns constantly teaches users to dismiss warnings unread, which is alert fatigue, and once that sets in the true positives stop working as well. The argument is therefore never "recall at any price". It is that one missed phishing site is worth some specific, finite number of false alarms. §5.3 fixes that number at 5 and shows what changes at 1, 3 and 10. The 5 is a judgement I make here in §1 on the cost structure above; the data has nothing to say about it.

**Assumptions this rests on**

1. The 30 features can be recomputed at request time, cheaply enough to block on. Most come straight from the URL and the page source, but `web_traffic`, `Page_Rank`, `Google_Index` and `Links_pointing_to_page` all require third-party lookups, so the deployment assumes those services are reachable and fast. Where they are not, the model is running on degraded input.
2. The warning is advisory rather than a hard block. The user can dismiss it and carry on. That is precisely what keeps the false-positive cost bounded. If the extension blocked outright, the whole cost argument changes and the threshold should move with it.
3. Phishing in 2015 still resembles phishing today. The dataset is a snapshot. Structural tricks such as hyphenated brand domains and deep subdomains have persisted, while others have not; §2.3 shows `Favicon`, `popUpWidnow` and `Iframe` are dead signals now. This is the weakest assumption in the project and I would rather state it than wait to be asked.
4. The attacker is not adapting to this particular model. Several features are cheap to flip once an attacker knows they are being measured, registering a longer-lived domain being the obvious example. A static classifier assumes a static adversary, which security never provides.
5. A warned user acts on the warning. If they click through regardless, recall buys nothing at all.

# 2. Data Understanding

## 2.1 Load dataset

Source: UCI Phishing Websites, id=327 (https://archive.ics.uci.edu/dataset/327/phishing+websites). Tutor-approved at Checkpoint 1. 11,055 rows x 30 features + target.

Loaded from `Training Dataset.arff`. **Not** `.old.arff` — that is a different, 2,456-row file from the same download that encodes binary features as {0,1} instead of {-1,1}, so it would silently change what every feature means.


In [ ]:
## Read the ARFF file into a pandas DataFrame
raw, meta = arff.loadarff('Training Dataset.arff')
df = pd.DataFrame(raw)

# scipy returns ARFF nominal values as bytes (b'-1'); convert to signed ints.
df = df.apply(lambda s: s.str.decode('utf-8')).astype(int)

TARGET = 'Result'
FEATURES = [c for c in df.columns if c != TARGET]

print(f'shape: {df.shape}  |  {len(FEATURES)} features')
df.head()


## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column
df.info()


In [ ]:
## Check for missing data
print('missing values:', df.isna().sum().sum())
print('duplicate rows:', df.duplicated().sum())


In [ ]:
## Describe data distribution
# Every column is a nominal code, so value counts are more informative than
# mean/std — a 'mean' of -0.3 on a {-1,1} column has no meaning.
cardinality = df[FEATURES].nunique().value_counts().rename('n_features')
print(cardinality.rename_axis('n_distinct_values'))

df.describe().T


The cardinality split determines how the EDA has to be done. All 30 features are small integer codes, 22 of them binary `{-1,1}` and 8 ternary `{-1,0,1}`, and not one is continuous. That removes most of the standard EDA and preprocessing repertoire.

Histograms and distribution plots have nothing to show. A histogram of a binary column is two bars; it reports the marginal split and says nothing about whether the feature separates the classes, which is the only question worth asking at this point.

Scaling and normalisation are unnecessary. The codes already sit near zero and share a common range, so no feature has a magnitude that could dominate a distance calculation or a gradient step. Applying `StandardScaler` here would change the numbers on screen without changing any model's behaviour.

There are no outliers to trim and nothing continuous to bin. Every value falls inside a known and very small level set, so "out of range" becomes a validation question, handled in the app in §6, rather than a cleaning one.

What takes their place is the conditional phishing rate per level. The quantity that carries information is P(phishing | feature = level) read against the 44.3% base rate, which is a crosstab rather than a histogram. A feature earns its place exactly when that rate moves as the level changes, and the size of that movement, the "spread" used below, is what §2.3.1.2 ranks on and what §5.1 either agrees with or has to explain away.

One consequence is worth flagging early. Thirty mostly-binary columns describe a very coarse space, so distinct websites can land on identical feature vectors without much difficulty. That is not hypothetical: §3.1 finds 11,055 rows collapsing to 5,785 unique vectors. The low cardinality observed here is the direct cause of the duplication problem there.

## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target
counts = df[TARGET].value_counts().sort_index()
labels = {-1: 'Phishing', 1: 'Legitimate'}

ax = sns.barplot(x=[labels[i] for i in counts.index], y=counts.values)
ax.set(title='Class balance of Result', xlabel='', ylabel='Websites')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,} ({v / len(df):.1%})', ha='center', va='bottom')
plt.show()

ratio = counts.max() / counts.min()
print(f'majority:minority = {ratio:.2f}:1')


The classes come close to balanced: 6,157 legitimate against 4,898 phishing, roughly 1.26:1.

What that rules out. This is not an imbalanced-classification problem. Resampling, SMOTE and `class_weight='balanced'` all treat a pathology that is absent here, so reaching for any of them would mean solving an imaginary problem. It also means accuracy is not broken as a metric, since a majority-class model scores about 55.7% rather than 95%, so accuracy still tells a useful model apart from a useless one. That 55.7% is the floor §4 has to clear.

What it means for the metric argument, which is the part that matters. Because the data sits near balance, the usual line is unavailable to me. I am not choosing recall because the data is imbalanced. I am prioritising recall because of the cost asymmetry laid out in §1: a false negative places a user on a credential-harvesting page, while a false positive costs one dismissed banner. Accuracy cannot see which of those two mistakes it made, and that blindness, rather than the class ratio, is why it is the wrong headline metric.

One thing to carry forward. This ratio is measured on the raw 11,055 rows. §3.1 deduplicates and the majority class flips to 51.6% phishing, which changes what the baseline scores and sharpens the point above considerably (§4.1).

### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features
# Every feature is a nominal code, so "distribution" on its own says little — a bar
# chart of a {-1,1} column just shows two bars. The question that carries information
# is conditional: *within each level of a feature, what share of sites are phishing?*
# The overall phishing rate is the reference; a level sitting far from it carries
# signal, a level sitting on it carries none.

BASE_RATE = (df[TARGET] == -1).mean()
print(f'overall phishing rate: {BASE_RATE:.1%}')


def phishing_rate_table(frame, features, target=TARGET):
    """One row per (feature, level): how many sites sit at that level, and what
    share of them are phishing."""
    rows = []
    for f in features:
        ct = pd.crosstab(frame[f], frame[target])
        for level, counts in ct.iterrows():
            n = counts.sum()
            rows.append({'feature': f, 'level': level, 'n': n,
                         'share_of_rows': n / len(frame),
                         'phishing_rate': counts.get(-1, 0) / n})
    return pd.DataFrame(rows)


levels = phishing_rate_table(df, FEATURES)
levels.to_csv(ARTIFACTS / 'feature_level_rates.csv', index=False)

# Discriminative power = how far apart a feature's own levels pull the phishing rate.
# This is a cheap univariate screen, not the feature selection itself (that is 5.1) —
# but it says up front which of the 30 columns could possibly matter.
spread = (levels.groupby('feature')['phishing_rate']
          .agg(lowest='min', highest='max')
          .eval('spread = highest - lowest')
          .sort_values('spread', ascending=False))

fig, ax = plt.subplots(figsize=(8, 9))
sns.barplot(x=spread['spread'], y=spread.index, ax=ax, color='#4C72B0')
ax.axvline(0.10, ls='--', c='crimson', lw=1)
ax.text(0.105, 0.99, '0.10 — rough "carries signal" line',
        transform=ax.get_xaxis_transform(), va='top', c='crimson', fontsize=9)
ax.set(title=f'Discriminative power of each feature\n'
             f'(spread in phishing rate across a feature\u2019s levels; overall rate {BASE_RATE:.1%})',
       xlabel='max \u2212 min phishing rate across levels', ylabel='')
plt.tight_layout()
plt.show()

spread.round(3)


In [ ]:
## Crosstabs: the six strongest features, and the three flattest for contrast
# Showing the flat ones matters as much as the strong ones — it is the evidence that
# some of the 30 columns are dead weight, which is what 5.1 acts on.

show = list(spread.index[:6]) + list(spread.index[-3:])

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharey=True)
for ax, f in zip(axes.ravel(), show):
    prop = pd.crosstab(df[f], df[TARGET], normalize='index')[[-1, 1]]
    prop.plot(kind='bar', stacked=True, ax=ax, legend=False,
              color=['#C44E52', '#55A868'], width=0.7)
    ax.axhline(BASE_RATE, ls='--', c='k', lw=1)
    ax.set(title=f'{f}  (spread {spread.loc[f, "spread"]:.2f})',
           xlabel='feature level', ylabel='share of sites')
    ax.tick_params(axis='x', rotation=0)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ['#C44E52', '#55A868']]
fig.legend(handles, ['Phishing', 'Legitimate'], loc='lower center', ncol=2)
fig.suptitle('Strongest six features vs the three flattest \u2014 dashed line = overall phishing rate')
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

# Level meanings are defined in `Phishing Websites Features.docx`; the numeric codes
# below are not self-explanatory and must be read against it before interpreting.
levels[levels['feature'].isin(show[:6])].round(3)


**Why the dominant two dominate**

`URL_of_Anchor` (spread 0.93) measures what fraction of the page's `<a>` tags point somewhere other than the page's own domain, whether to `#`, to `javascript:void(0)`, or off-site entirely. That comes close to a structural definition of "this page is a copy of another site". A phishing page is a visual clone whose navigation links generally still point at the real brand's website, because that is what makes it look right and it costs far less than rebuilding the whole site. A genuine site links mostly to itself. The feature is therefore not a heuristic about phishing; it detects the cloning method itself, which is why 99% of sites at its worst level are phishing and only 6% at its best.

`SSLfinal_State` (spread 0.87) encodes certificate presence, issuer trust and certificate age in a single code. Phishing infrastructure is short-lived and disposable, whereas a year-old certificate from a trusted CA implies both money and a domain-validation step tied to a real identity, neither of which suits a domain intended to last a week. The interesting part is that the feature is not monotone (see §5.3): the "suspicious" middle level runs 98.2% phishing, worse than having no HTTPS at all at 85.8%. That follows once free DV certificates made a padlock cheap. An untrusted or brand-new certificate is a deliberate attempt to look secure, and the intent is more incriminating than not bothering. It serves as a reminder that the coded levels carry whatever meaning the data gives them rather than the meaning the documentation implies.

Between them these two nearly separate the classes on their own. The model should lean on them and does; both survive selection in §5.1 comfortably.

**The middle tier: eight features that describe the attack, not the attacker**

Each of these carries spread 0.24 to 0.51, which is real signal, not decisive alone, and mostly explainable as a side-effect of how a phishing page has to be built.

`Prefix_Suffix` picks up a hyphen in the domain, as in `paypal-secure.com`. The brand name itself is unavailable, so the attacker appends to it.

`having_Sub_Domain` covers `paypal.com.login.attacker.com`. It puts the brand where a user scanning left to right will see it while the registrable domain belongs to the attacker.

`SFH`, the server form handler, records where the login form actually POSTs. A phishing form has to exfiltrate credentials somewhere off-domain or to a blank handler. As with `URL_of_Anchor`, this one sits on the attack mechanism rather than near it.

`Request_URL` captures what fraction of images, scripts and CSS load from another domain. A clone hot-links the real site's assets instead of copying them.

`Links_in_tags` applies the same off-domain logic to `<meta>`, `<script>` and `<link>`.

`web_traffic` is Alexa rank. A domain registered last week has no traffic history. This is a reputation signal rather than a content one, and it is the hardest of the set for an attacker to fake.

`URL_Length` matters because long URLs push the real domain out of the visible part of the address bar and carry obfuscation.

`Domain_registeration_length` needs flagging. The documentation codes 1 as "registered for more than a year", supposedly the legitimate level, yet in this data that level runs 60.2% phishing against 36.4% for short registration (§5.3). The encoding runs backwards from intuition. I should not read it as "young domain implies phishing" simply because that is the story that sounds right.

**The three dead features, and why they are dead**

`Favicon` (spread 0.000), `popUpWidnow` (0.0001) and `Iframe` (0.006) show a phishing rate equal to the base rate at every level, so knowing them tells me nothing whatsoever. That is obsolescence rather than a data problem. These were 2015-era tells that have since either become universal, as every site now has a favicon, or been neutralised by browsers, since pop-ups and cross-origin iframes are blocked by default and nobody bothers. A signal discriminates only while it remains unusual.

A prediction, recorded now so §5.1 can be checked against it: all three should finish at the bottom of both the RandomForest-importance and mutual-information rankings. If they do not, that disagreement needs explaining rather than ignoring.

One caveat. Flat is not the same as skewed. `Prefix_Suffix` has 87% of its rows in a single level and still reaches spread 0.51. A feature can be rare and highly informative when it fires. I am dropping these three for having no signal, not for being lopsided.

### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables
# Spearman rather than Pearson: these are ordered nominal codes, not measured
# quantities, so rank agreement is the only correlation that means anything here.

corr = df[FEATURES].corr(method='spearman')

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap='vlag', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.4, cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Spearman correlation between features (lower triangle)')
plt.tight_layout()
plt.show()


In [ ]:
## The pairs behind the heatmap, plus each feature's correlation with the target
pairs = (corr.where(~mask).stack().rename('corr').reset_index()
         .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'}))
pairs['abs_corr'] = pairs['corr'].abs()
top_pairs = pairs.nlargest(8, 'abs_corr')[['feature_a', 'feature_b', 'corr']]

# A correlation between two near-constant columns is driven by a handful of rows, so
# it reflects shared skew rather than shared information. Carrying each feature's
# majority-level share here stops the heatmap being read as "these are redundant"
# when the truth is "these are both almost always the same value".
skew = levels.groupby('feature')['share_of_rows'].max().rename('majority_level_share')
top_pairs = (top_pairs
             .merge(skew, left_on='feature_a', right_index=True)
             .merge(skew, left_on='feature_b', right_index=True, suffixes=('_a', '_b')))

print('Most correlated feature pairs:')
print(top_pairs.round(3).to_string(index=False))

print('\nSpearman correlation with the target (top 10 by magnitude):')
target_corr = (df[FEATURES].corrwith(df[TARGET], method='spearman')
               .sort_values(key=abs, ascending=False))
print(target_corr.head(10).round(3).to_string())


The strong pairs are an artefact of skew, and I am not treating them as redundancy.

Every feature appearing in the top correlated pairs, namely `popUpWidnow`↔`Favicon` (ρ = 0.94), `double_slash_redirecting`↔`Shortining_Service` (0.84), `port`↔`Favicon` (0.80) and `Submitting_to_email`↔`port` (0.80), is one of the heavily skewed columns, with 86% to 96% of rows sitting in a single level. When two columns are almost always 1, the correlation coefficient is decided by the few hundred rows where either one differs, so they will look correlated whatever they actually measure. A high ρ between two near-constant columns says something about their variance rather than about shared information.

So I will do nothing about them here. Dropping one of each pair on correlation grounds would arrive at the right outcome for the wrong reason. These features do get dropped, but in §5.1, for having no measurable signal, since §2.3.1.2 shows their conditional phishing rate flat at every level. Removing them at this stage would also mean making a modelling decision inside EDA on evidence that cannot support it.

The more useful finding is that the informative tier has no multicollinearity to fix. `SSLfinal_State` and `URL_of_Anchor` correlate 0.74 and 0.70 with the target but not with each other, so they carry independent evidence, one about certificate identity and the other about page structure, rather than two views of the same thing. Two consequences follow.

For `LogisticRegression` (§4) there is no correlated-predictor problem, so the coefficients stay stable and interpretable and I do not need regularisation to control variance inflation.

For the feature-selection plan (§5.1) nothing changes. Selection runs on RandomForest importance and mutual information against the target rather than on pruning correlated pairs. Had the two dominant features been near duplicates of one another I would have needed to handle it; they are not, so both should survive, and both do.

# 3. Data Preparation

## 3.1 Data Cleaning

The 30 features were extracted from raw URLs and page HTML by the original researchers and are already encoded as nominal codes. There is no raw URL string in this dataset, so there is nothing to engineer *from*.

**This project therefore uses feature selection, not feature engineering.** The spec asks for "feature engineering and/or feature selection", so selection alone is compliant — inventing transformations of already-engineered codes would be noise dressed up as work. Selection is done in section 5.1.


**Duplicates — the one real data-quality problem in this dataset.**

There are no missing values, but 11,055 rows collapse to only **5,785 unique feature vectors** (47% of rows are exact duplicates of another row). Two consequences:

1. **Leakage.** With a random train/test split, roughly half the test rows have an identical twin sitting in the training set. A tree model can memorise the twin and 'predict' it perfectly, so the reported test scores would be optimistic — they would measure recall on rows the model has effectively already seen.
2. **A hard accuracy ceiling.** 64 feature vectors appear with *both* labels (357 rows). Those are genuinely indistinguishable given these 30 features, so no model can get all of them right. That is irreducible error, not a modelling failure.

This matters more than anything else in cleaning, and it is not obvious from `df.info()` — it only shows up if you check.


In [ ]:
## Quantify the duplication before deciding what to do about it
print(f'rows                      {len(df):,}')
print(f'unique feature vectors    {df[FEATURES].drop_duplicates().shape[0]:,}')
print(f'unique rows incl. label   {df.drop_duplicates().shape[0]:,}')

conflicts = df.groupby(FEATURES)[TARGET].nunique()
print(f'feature vectors with both labels: {(conflicts > 1).sum()}')


**Decision: drop the duplicates. `df_dedup` (5,849 rows) is canonical from here on.**

What it costs me, stated up front. The dataset roughly halves, from 11,055 rows to 5,849, and the test set falls to 1,170 rows, so every rate I report afterwards carries something like ±1pp of sampling noise. I accept that, and I stop quoting four decimal places as though all four were meaningful. For 30 low-cardinality features, 5,849 rows remains ample.

The finding that settles it is that the inflation is model-specific. `LogisticRegression` and `GradientBoosting` barely move when duplicates come out, with recall within 0.2pp. Only `RandomForest` is meaningfully inflated: recall 0.9612 → 0.9404 and accuracy 0.9742 → 0.9419. The mechanism is straightforward. An unpruned forest grows leaves fine enough to memorise individual training rows, so a test row with an identical twin in training gets answered from memory rather than from generalisation. A linear model has no capacity to store a single row and a shallow-stage booster has very little. The duplicates therefore reward one model family in particular.

The corroborating evidence is what I would lead with in a viva. `RandomForest`'s train to test gap widens from 0.016 to 0.049 once the duplicates are gone. Deduplication did not damage `RandomForest`; it exposed overfitting the duplicates had been hiding. The general claim that "leakage inflates all scores" is therefore wrong here, and the specific claim is more interesting: leakage inflates whichever model has the capacity to memorise, and conceals that model's overfitting while doing it.

How bad the leakage is. With duplicates kept, 65.8% of test rows have a feature-identical twin in the training set; deduplicated, that falls to 1.8%. A "test score" on the kept data is therefore mostly a score on rows the model has already seen, which is no estimate of generalisation at all.

Rejecting the counter-argument. The standard defence holds that duplicate rows are frequency information and belong in the model as sample weights. That does not apply here. These are not repeat observations of the same website. They are different websites colliding onto the same vector because 30 mostly-binary codes describe a very coarse space, exactly as §2.2 predicted. A collision caused by low feature resolution is not evidence about how common a configuration is in the wild; it is an artefact of the encoding. Weighting by it would amount to weighting by how lossy the feature set happens to be.

The class balance flips, which is useful rather than alarming. With duplicates kept the data is 55.7% legitimate; deduplicated it is 51.6% phishing. `DummyClassifier` therefore flips from always predicting legitimate, with phishing recall 0.00, to always predicting phishing, with phishing recall 1.00 and accuracy 0.52. A do-nothing model now scores a perfect 1.000 on the very metric this project claims to prioritise. That is the cleanest available demonstration of why recall cannot be reported alone, and why §5 reports precision, recall and F1 together. I use it again there.

The 64 contradictory vectors stay deliberately. `drop_duplicates()` runs over the features together with the label, so the 64 vectors that appear under both labels survive as one row per label, giving 128 rows or 2.2% of the deduplicated set. They represent genuine ambiguity: two real websites, identical under these 30 features, one phishing and one not. Deleting them would manufacture a ceiling the model cannot actually reach and would make the final scores look better than the feature set deserves. They are the reason §5.3 does not report 99% and should not be expected to.

In [ ]:
## Clean data
# There is no missing data and no out-of-range codes, so cleaning is exactly one
# decision: what to do about the 5,206 duplicate rows.
#
# Rather than assert which choice is right, build BOTH variants and let section 4
# measure the difference. `df_dedup` is the canonical dataset carried through 5.x;
# `df_kept` exists only to quantify what keeping duplicates would have bought.

df_kept = df.copy()
df_dedup = df.drop_duplicates().reset_index(drop=True)

print(f'duplicates kept : {len(df_kept):,} rows')
print(f'deduplicated    : {len(df_dedup):,} rows  '
      f'({1 - len(df_dedup) / len(df_kept):.1%} removed)')

# drop_duplicates() is over feature+label together, so the 64 contradictory vectors
# survive as one row per label. That is deliberate: they are genuine ambiguity in the
# feature space, not dirt, and deleting them would fake a ceiling the model cannot reach.
conflicts_after = df_dedup.groupby(FEATURES)[TARGET].nunique()
print(f'contradictory vectors surviving dedup: {(conflicts_after > 1).sum()} '
      f'({(conflicts_after > 1).sum() * 2:,} rows, '
      f'{(conflicts_after > 1).sum() * 2 / len(df_dedup):.1%} of the deduped data)')

# Deduplication is not label-neutral — check before assuming the class ratio survived.
balance = pd.DataFrame({
    'duplicates kept': df_kept[TARGET].value_counts(normalize=True),
    'deduplicated': df_dedup[TARGET].value_counts(normalize=True),
}).rename(index={-1: 'phishing', 1: 'legitimate'})
print()
print(balance.round(3).to_string())


## 3.2 Train-Test Split

In [ ]:
## Split data into train set and test set
# Stratified so each test set keeps its own real class ratio — otherwise recall on the
# phishing class is measured against a distribution that does not exist.

def make_split(frame):
    return train_test_split(frame[FEATURES], frame[TARGET], test_size=0.2,
                            stratify=frame[TARGET], random_state=RANDOM_STATE)


splits = {'duplicates kept': make_split(df_kept), 'deduplicated': make_split(df_dedup)}

# The canonical split, used everywhere from 5.1 onward.
X_train, X_test, y_train, y_test = splits['deduplicated']
print(f'canonical (deduplicated): train {X_train.shape}  test {X_test.shape}')


In [ ]:
## Measure the leakage directly instead of asserting it
# The argument against keeping duplicates is that test rows reappear in training. That
# is checkable: what share of test rows have a feature-identical twin in the train set?

print('share of test rows with an identical feature vector in train:')
for name, (X_tr, X_te, _, _) in splits.items():
    train_vectors = set(map(tuple, X_tr.to_numpy()))
    overlap = np.mean([tuple(row) in train_vectors for row in X_te.to_numpy()])
    print(f'  {name:16s} {overlap:6.1%}')


# 4. Modelling

## 4.1 Baseline

`DummyClassifier(strategy='most_frequent')` always predicts the majority class. Every model below has to beat it, otherwise it has learned nothing. The rubric's A band for model comparison requires test-set results *with* a baseline comparison.


In [ ]:
## Majority-class baseline
# Fitted on both variants, because deduplication changes which class is the majority
# and therefore changes what this baseline even predicts.

for name, (X_tr, X_te, y_tr, y_te) in splits.items():
    dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
    pred = dummy.predict(X_te)
    print(f'{name:16s} predicts {dummy.classes_[dummy.predict(X_te[:1])[0] == dummy.classes_][0]:>2} '
          f'| accuracy {accuracy_score(y_te, pred):.4f} '
          f'| phishing recall {recall_score(y_te, pred, pos_label=-1, zero_division=0):.4f}')


### 4.2 Train Model

Three distinct algorithm families, all on the full 30 features (the 'before' half of the feature-selection comparison in 5.1):

- **LogisticRegression** — linear, coefficients are directly readable
- **RandomForestClassifier** — bagged trees, captures interactions
- **GradientBoostingClassifier** — boosted trees, usually strongest, slowest to train

scikit-learn only, per the spec.


In [ ]:
## Initialise and train models
# Same four estimators, both dataset variants, all 30 features. Building the models
# fresh per dataset avoids refitting an already-fitted instance.


def build_models():
    return {
        'DummyClassifier': DummyClassifier(strategy='most_frequent'),
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    }


# pos_label=-1 throughout: "positive" means *phishing*, the class whose misses are the
# expensive ones. Getting this backwards silently reports the wrong recall.
rows = []
for dataset, (X_tr, X_te, y_tr, y_te) in splits.items():
    for name, model in build_models().items():
        t0 = time.perf_counter()
        model.fit(X_tr, y_tr)
        fit_seconds = time.perf_counter() - t0
        pred_te, pred_tr = model.predict(X_te), model.predict(X_tr)
        rows.append({
            'dataset': dataset, 'model': name,
            'test_accuracy': accuracy_score(y_te, pred_te),
            'test_precision': precision_score(y_te, pred_te, pos_label=-1, zero_division=0),
            'test_recall': recall_score(y_te, pred_te, pos_label=-1, zero_division=0),
            'test_f1': f1_score(y_te, pred_te, pos_label=-1, zero_division=0),
            'train_accuracy': accuracy_score(y_tr, pred_tr),
            'fit_seconds': fit_seconds,
        })

baseline_metrics = pd.DataFrame(rows)
# Train minus test accuracy: the overfitting evidence for the qualitative comparison.
baseline_metrics['train_test_gap'] = (baseline_metrics['train_accuracy']
                                      - baseline_metrics['test_accuracy'])
baseline_metrics.to_csv(ARTIFACTS / 'baseline_metrics.csv', index=False)

baseline_metrics.round(4)


In [ ]:
## The duplicate decision, quantified
# Side-by-side on the metric that matters (phishing recall) plus accuracy, so the cost
# of keeping duplicates is a number rather than an assertion.

for metric in ['test_recall', 'test_accuracy', 'train_test_gap']:
    table = baseline_metrics.pivot(index='model', columns='dataset', values=metric)
    table['difference'] = table['duplicates kept'] - table['deduplicated']
    print(f'\n{metric}:')
    print(table.round(4).to_string())


On the canonical (deduplicated) data, all 30 features:

| Model | Test recall | Test F1 | Test acc | Train−test gap | Fit time |
| --- | --- | --- | --- | --- | --- |
| DummyClassifier | 1.000 | 0.681 | 0.516 | −0.000 | 0.000s |
| LogisticRegression | 0.904 | 0.915 | 0.913 | 0.011 | 0.013s |
| RandomForest | 0.940 | 0.944 | 0.942 | **0.049** | 0.197s |
| GradientBoosting | 0.940 | 0.948 | **0.946** | 0.006 | 0.793s |

**Carrying `GradientBoosting` into tuning.**

Quantitatively, `RandomForest` and `GradientBoosting` tie exactly on phishing recall at 0.9404, and GB comes marginally ahead on F1 (0.948 against 0.944) and accuracy (0.946 against 0.942). Those margins sit inside sampling noise on a 1,170-row test set, so the metrics table alone would leave me choosing on a coin flip. That is why the third-decimal comparison is not the reason I am giving.

The reason I am giving is the train to test gap: 0.006 for GB against 0.049 for RF, a factor of eight. Two models reporting the same recall are not equally believable if one of them is memorising to get there. §3.1 already caught `RandomForest` doing exactly that on duplicate rows, and its gap grew after deduplication. GB reaching the same recall with a gap near zero is the stronger result, and it makes GB's test number more likely to survive contact with data from a different month.

Qualitatively, on the four axes that matter for this deployment:

| | `LogisticRegression` | `RandomForest` | `GradientBoosting` |
| --- | --- | --- | --- |
| Interpretability | best; signed coefficients read directly as "this feature pushes toward phishing" | impurity importances only, unsigned | same as RF, plus staged structure is harder to narrate |
| Fit time | 0.013s | 0.197s | 0.793s |
| Overfitting risk | lowest (gap 0.011) | highest (0.049) | lowest of the two tree models (0.006) |
| Cost to serve in the app | trivial | trivial | trivial |

On fit time, GB is 4× RF and 60× LR, and none of it matters. Training happens once, offline, and finishes in under a second. What the Streamlit app pays for is inference on a single row, which is microseconds for all three, plus a pickle small enough to load on every cold start. "Slowest to train" and "worst to deploy" are different claims and I should not let one stand in for the other.

The deployment need that actually decides it. §5.3 does not use the default 0.5 cut-off; it moves the decision threshold on `predict_proba` to buy recall at a known cost in precision. That requires a model whose probability outputs are sensibly ordered and spread across the range, so that moving the threshold trades the two errors smoothly instead of in jumps. Gradient boosting on log-odds provides that, whereas `RandomForest`'s probabilities are vote proportions over a small number of discrete outcomes and come out coarser. Since the threshold is the main lever in the whole project (§5.3), the model that supports it best wins on more than tiebreak grounds.

Why not `LogisticRegression`, honestly. It gives up 3.6pp of recall, roughly 22 extra missed phishing sites per 604 in the test set, and buys readable coefficients in exchange. §1 identifies the missed phishing site as the expensive error, and the app never shows a coefficient to a user, so interpretability is worth less here than it would be somewhere each individual decision has to be justified. It stays in the notebook as the baseline establishing what a linear decision boundary can do on this feature set, which turns out to be most of it. That is a finding in its own right: the tree models are only about 3pp better, so the problem is close to linearly separable.

The dummy is why recall alone cannot be the criterion. All three real models beat `DummyClassifier` on F1 (0.681) and accuracy (0.516), but only two beat it on recall, because its recall is 1.000 by construction on the deduplicated split. Any criterion a do-nothing model wins outright is not a criterion. §5 uses this.

# 5. Model Evaluation

## Evaluation metric selection

**What the trade actually costs, measured on the test set (604 phishing, 566 legitimate):**

| | Threshold 0.50 | Threshold 0.08 (FN worth 5 FP) |
| --- | --- | --- |
| Phishing recall | 0.9421 | **0.9901** |
| Precision | 0.9421 | 0.8592 |
| Phishing sites missed | 35 | **6** |
| False alarms | 35 | 98 |

What I optimise, in one line: I fit and select on F1, then move the decision threshold to buy recall at a stated price, and I report precision, recall, F1 and the confusion matrix together throughout so that the trade stays visible rather than asserted.

Why not accuracy. Imbalance is not the reason, since the deduplicated split is 51.6% phishing against 48.4% legitimate and accuracy is perfectly capable of separating a good model from a useless one here. The problem is that accuracy cannot see which mistake was made. §1 establishes that the two mistakes are not comparable: a false negative puts a user on a credential-harvesting page and is both unbounded and irreversible, while a false positive costs one dismissed banner. A metric that adds those together at 1:1 has quietly made a cost assumption I explicitly reject. Saying "the data was imbalanced so I used recall" would be wrong on this dataset and would be exactly the claim the demo goes looking for.

Why not recall. Because it is a dial rather than a metric. `DummyClassifier` in §4.1 scores recall 1.000 on this split by predicting phishing every time. Any quantity a do-nothing model wins outright cannot be the thing being optimised, and a model selected to maximise recall is a model selected to approach that dummy.

Why F1 for fitting and selection. It is the metric no degenerate model can win in either direction: the always-phishing dummy scores 0.681 on it and an always-legitimate model would score 0. That makes it safe as the selection criterion in §5.1 for choosing k and §5.2 for choosing hyperparameters, where I need a single number and cannot afford to let the choice be gamed. It does not encode my cost asymmetry, which is deliberate. I would rather apply the cost preference once, explicitly and visibly, than smuggle it into every intermediate decision.

Why the threshold is where the cost preference goes. Hyperparameters move precision and recall together and unpredictably, as §5.2 demonstrates. The decision threshold on `predict_proba` trades them against one another on purpose, monotonically, with a number I can state and defend. It converts §1's cost argument into an operational setting instead of a decorative paragraph, and it means anyone who disagrees with my 5:1 assumption can change one constant and watch what happens rather than retraining anything.

What the trade costs, stated as a claim I will defend: moving from 0.50 to 0.08 catches 29 more phishing sites, taking misses from 35 to 6, and produces 63 more false alarms, going from 35 to 98. At §1's 5:1 valuation that is a good trade by a factor of just over two; at 1:1 it is a bad one. The defensibility of the whole operating point therefore rests on the cost argument in §1 rather than on anything in the data, and I should say so before anyone asks.

In [ ]:
## Evaluate model
# The model section 4 selected, on all 30 features, before any of the 5.x iteration.
# This is the "before" that 5.3's final evaluation is measured against, so it is scored
# the same way: fitted once on the training split, predicted once on the test split.

section4_model = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
section4_pred = section4_model.predict(X_test)

print('GradientBoosting, all 30 features (the section 4 selection)\n')
print(classification_report(y_test, section4_pred, labels=[-1, 1],
                            target_names=['phishing (-1)', 'legitimate (1)'], digits=4))

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ConfusionMatrixDisplay.from_predictions(
    y_test, section4_pred, labels=[-1, 1],
    display_labels=['phishing', 'legitimate'], cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Section 4 model: 30 features, threshold 0.50')
plt.show()


## Iterative model development

Order matters: baselines on all 30 features (section 4) come first, so 5.1 has a genuine 'before' to compare against.


### 5.1 Feature selection

This is **feature selection, not feature engineering** — for the reason given in §3.1: the 30 columns were already extracted from raw URLs and HTML by the original researchers, and no raw URL string survives in the dataset to engineer new features from.

Two independent methods, cross-checked against each other:

1. **RandomForest feature importances** — model-based, captures interactions
2. **Mutual information** (`mutual_info_classif`, the criterion behind `SelectKBest`) — statistical, no model assumed; mutual information suits categorical features where a linear correlation would not

Agreement between two different methods is stronger evidence than either alone.

**How many features survive is decided before looking at the test set.** Both rankings and the cut are computed on the *training* split only; the size of the surviving set comes from 5-fold cross-validation *within* the training data, using the one-standard-error rule — keep the smallest feature set whose cross-validated score is within one standard error of the best one seen. Choosing `k` by re-scoring the test set would make the before/after comparison below meaningless, because the "after" would have been fitted to the very data it is reported on.


In [ ]:
## Further feature selection
# Both rankings are computed on the TRAINING split only. Ranking on the full dataset
# would let test rows influence which features exist, which is leakage by another route.
#
#   1. RandomForest impurity importances - model-based, sees interactions.
#   2. Mutual information via SelectKBest - model-free. discrete_features=True matters:
#      every column is a nominal code, so estimating MI as if -1/0/1 were points on a
#      continuous scale would be measuring the wrong thing.
#
# Neither is trusted alone. The surviving set is ordered by the two ranks averaged, and
# the two are cross-checked against each other first.

rf_ranker = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

# k='all' on purpose: SelectKBest is used here for its *scores*, not to pick the size of
# the set. Passing a k would be inventing the number this section exists to measure -
# the 1-SE cross-validation rule below decides it instead.
mi_selector = SelectKBest(
    score_func=partial(mutual_info_classif, discrete_features=True,
                       random_state=RANDOM_STATE),
    k='all').fit(X_train, y_train)

ranking = pd.DataFrame({
    'rf_importance': rf_ranker.feature_importances_,
    # Spread of the importance across the 100 trees: a large std means the forest does
    # not agree with itself about that feature, so the point estimate is not solid.
    'rf_std': np.std([t.feature_importances_ for t in rf_ranker.estimators_], axis=0),
    'mutual_info': mi_selector.scores_,
}, index=FEATURES)
ranking['rf_rank'] = ranking['rf_importance'].rank(ascending=False)
ranking['mi_rank'] = ranking['mutual_info'].rank(ascending=False)
ranking['mean_rank'] = ranking[['rf_rank', 'mi_rank']].mean(axis=1)
ranking = ranking.sort_values('mean_rank')

rho = spearmanr(ranking['rf_rank'], ranking['mi_rank']).statistic
print(f'Spearman rank correlation between the two methods: {rho:.3f}')
print('\nlargest disagreements (|rf_rank - mi_rank|):')
print((ranking['rf_rank'] - ranking['mi_rank']).abs().sort_values(ascending=False)
      .head(4).to_string())

# Section 2 predicted, from conditional phishing rates alone, that Favicon, popUpWidnow
# and Iframe carry no signal. Two model-based rankings are now in a position to falsify
# that. Printed explicitly rather than quietly hoped for.
print('\nS2 predicted three dead features. Where did the two rankings put them?')
print(ranking.loc[['Favicon', 'popUpWidnow', 'Iframe'],
                  ['rf_rank', 'mi_rank', 'mean_rank']].to_string())

ranking.to_csv(ARTIFACTS / 'feature_ranking.csv')
ranking.round(4)


In [ ]:
## Plot both rankings side by side
# Same feature order (mean rank) on both panels, so agreement and disagreement between
# the two methods is read off vertically instead of by comparing two sorted lists.

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)
order = ranking.index[::-1]  # matplotlib draws barh bottom-up

axes[0].barh(order, ranking.loc[order, 'rf_importance'],
             xerr=ranking.loc[order, 'rf_std'], color='#4C72B0',
             error_kw={'elinewidth': 0.8, 'ecolor': '#666666'})
axes[0].set_title('RandomForest importance\n(bars = mean, whiskers = std across trees)')
axes[0].set_xlabel('impurity decrease')

axes[1].barh(order, ranking.loc[order, 'mutual_info'], color='#DD8452')
axes[1].set_title('Mutual information\n(model-free)')
axes[1].set_xlabel('bits shared with the target')

fig.suptitle('Two independent feature rankings, ordered by their average rank', y=0.995)
fig.tight_layout()
plt.show()


In [ ]:
## How many features to keep - decided by cross-validation, not by taste
# Walk k = 1..30 down the averaged ranking and cross-validate inside the training split.
# All three models are swept, not just the one carried forward: if the curves plateau in
# the same place, the cut is a property of the data, not of one estimator's quirks.
#
# Takes ~100s - it is 30 x 3 x 5 = 450 fits, and GradientBoosting is the slow one.

ordered_features = list(ranking.index)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'recall': make_scorer(recall_score, pos_label=-1, zero_division=0),
           'f1': make_scorer(f1_score, pos_label=-1, zero_division=0)}

rows = []
for k in range(1, len(ordered_features) + 1):
    cols = ordered_features[:k]
    for name, model in build_models().items():
        if name == 'DummyClassifier':
            continue
        cvres = cross_validate(model, X_train[cols], y_train, cv=cv, scoring=scoring)
        rows.append({'k': k, 'model': name,
                     'cv_recall': cvres['test_recall'].mean(),
                     'cv_recall_se': cvres['test_recall'].std(ddof=1) / np.sqrt(cv.get_n_splits()),
                     'cv_f1': cvres['test_f1'].mean(),
                     'cv_f1_se': cvres['test_f1'].std(ddof=1) / np.sqrt(cv.get_n_splits())})

sweep = pd.DataFrame(rows)
sweep.to_csv(ARTIFACTS / 'feature_sweep.csv', index=False)


def smallest_k_within_1se(frame, mean_col, se_col):
    """One-standard-error rule: of every k whose mean score reaches (best - 1 SE of the
    best), take the smallest. Prefers the simpler model when the difference is inside
    the noise of the cross-validation itself."""
    best = frame.loc[frame[mean_col].idxmax()]
    return int(frame.loc[frame[mean_col] >= best[mean_col] - best[se_col], 'k'].min()), best


print('smallest k within 1 SE of that model\'s own best CV score:')
for name in sweep['model'].unique():
    s = sweep[sweep.model == name]
    k_f1, b_f1 = smallest_k_within_1se(s, 'cv_f1', 'cv_f1_se')
    k_re, b_re = smallest_k_within_1se(s, 'cv_recall', 'cv_recall_se')
    print(f'  {name:20s} by F1: k={k_f1:2d}  (best {b_f1.cv_f1:.4f} at k={int(b_f1.k)})'
          f'   | by recall: k={k_re:2d}  (best {b_re.cv_recall:.4f} at k={int(b_re.k)})')

# The model carried forward from section 4.2. One constant, deliberately: if the
# argument in 4.2 lands on a different model, change it here and 5.1-5.3 follow.
SELECTED_MODEL = 'GradientBoosting'

# Cut on F1, not on recall alone. Recall by itself would pick a much smaller set (see
# above) because recall can be bought by predicting 'phishing' more often - the dummy in
# section 4.1 scores 1.000 on it. F1 keeps precision in the objective, so the surviving
# set has to earn its size on both classes.
selected_curve = sweep[sweep.model == SELECTED_MODEL]
K, best = smallest_k_within_1se(selected_curve, 'cv_f1', 'cv_f1_se')
SELECTED_FEATURES = ordered_features[:K]
DROPPED_FEATURES = ordered_features[K:]

print(f'\nselected model : {SELECTED_MODEL}')
print(f'best CV F1     : {best.cv_f1:.4f} at k={int(best.k)}, SE {best.cv_f1_se:.4f}'
      f'  -> keep any k scoring >= {best.cv_f1 - best.cv_f1_se:.4f}')
print(f'chosen k       : {K}   ({len(DROPPED_FEATURES)} features dropped)')
print(f'\nkept    : {SELECTED_FEATURES}')
print(f'\ndropped : {DROPPED_FEATURES}')

json.dump({'selected_model': SELECTED_MODEL, 'k': K,
           'selection_rule': 'smallest k within 1 SE of best 5-fold CV F1 on the '
                             'training split, features ordered by mean(RF importance '
                             'rank, mutual information rank)',
           'features': SELECTED_FEATURES, 'dropped': DROPPED_FEATURES},
          open(ARTIFACTS / 'selected_features.json', 'w'), indent=2)


In [ ]:
## The two curves that justify the cut
# Left: every model's F1 against k, so the plateau can be seen to be shared.
# Right: recall vs F1 for the selected model - they saturate at very different k, which
# is the whole reason the cut is made on F1.

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name in sweep['model'].unique():
    s = sweep[sweep.model == name]
    axes[0].plot(s['k'], s['cv_f1'], marker='o', ms=3, label=name)
axes[0].axhline(best.cv_f1 - best.cv_f1_se, ls=':', c='grey',
                label=f'{SELECTED_MODEL} best - 1 SE')
axes[0].axvline(K, ls='--', c='crimson', label=f'chosen k = {K}')
axes[0].set(xlabel='number of features kept (k)', ylabel='5-fold CV F1 (phishing)',
            title='All three models plateau in the same region')
axes[0].legend(fontsize=8)

s = selected_curve
axes[1].plot(s['k'], s['cv_recall'], marker='o', ms=3, label='recall', c='#C44E52')
axes[1].plot(s['k'], s['cv_f1'], marker='o', ms=3, label='F1', c='#4C72B0')
axes[1].axvline(K, ls='--', c='crimson')
axes[1].set(xlabel='number of features kept (k)', ylabel='5-fold CV score',
            title=f'{SELECTED_MODEL}: recall saturates long before F1 does')
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()


In [ ]:
## Before / after: all 30 features vs reduced set
# Refit on the reduced set and score the test set once. The 'before' half is read back
# from section 4's saved metrics rather than recomputed, so the two halves are
# guaranteed to come from the same split and the same fitted-once protocol.

before = pd.read_csv(ARTIFACTS / 'baseline_metrics.csv')
before = before[(before.dataset == 'deduplicated') & (before.model != 'DummyClassifier')]

rows = []
for name, model in build_models().items():
    if name == 'DummyClassifier':
        continue
    t0 = time.perf_counter()
    model.fit(X_train[SELECTED_FEATURES], y_train)
    fit_seconds = time.perf_counter() - t0
    pred_te = model.predict(X_test[SELECTED_FEATURES])
    pred_tr = model.predict(X_train[SELECTED_FEATURES])
    rows.append({
        'model': name,
        'test_accuracy': accuracy_score(y_test, pred_te),
        'test_precision': precision_score(y_test, pred_te, pos_label=-1, zero_division=0),
        'test_recall': recall_score(y_test, pred_te, pos_label=-1, zero_division=0),
        'test_f1': f1_score(y_test, pred_te, pos_label=-1, zero_division=0),
        'train_accuracy': accuracy_score(y_train, pred_tr),
        'fit_seconds': fit_seconds,
    })

reduced_metrics = pd.DataFrame(rows)
reduced_metrics['train_test_gap'] = (reduced_metrics['train_accuracy']
                                     - reduced_metrics['test_accuracy'])
reduced_metrics['n_features'] = K
reduced_metrics.to_csv(ARTIFACTS / 'reduced_metrics.csv', index=False)

COMPARE_ON = ['test_recall', 'test_precision', 'test_f1', 'test_accuracy', 'train_test_gap']
comparison = (before.set_index('model')[COMPARE_ON]
              .join(reduced_metrics.set_index('model')[COMPARE_ON],
                    lsuffix=' (30)', rsuffix=f' ({K})'))

print(f'canonical deduplicated test set - 30 features vs {K}:')
for metric in COMPARE_ON:
    pair = comparison[[f'{metric} (30)', f'{metric} ({K})']].copy()
    pair['change'] = pair[f'{metric} ({K})'] - pair[f'{metric} (30)']
    print(f'\n{metric}:')
    print(pair.round(4).to_string())


Test set, `GradientBoosting`, 30 features → 16:

| Metric | 30 features | 16 features | Change |
| --- | --- | --- | --- |
| Recall (phishing) | 0.9404 | 0.9387 | −0.0017 |
| Precision | 0.9546 | 0.9562 | +0.0015 |
| F1 | 0.9475 | 0.9474 | −0.0001 |
| Accuracy | 0.9462 | 0.9462 | 0.0000 |
| Train−test gap | 0.0062 | **0.0013** | −0.0049 |

**Performance held. It did not improve, and that is the correct claim to make.**

F1 moved by one ten-thousandth, 0.9475 → 0.9474, and accuracy did not move at all. Claiming feature selection "improved the model" would mean reading noise as a result. The honest headline is the same performance on roughly half the inputs, and the two things that genuinely changed are worth more than a metric bump would have been.

1. The train to test gap tightened fivefold, 0.0062 → 0.0013. A smaller hypothesis space generalises more tightly. This is the third occasion in the project where model capacity, rather than data or hyperparameters, turned out to be what drives the gap; see §3.1 for `RandomForest` and §5.2 for the tuned booster.
2. Fourteen fewer questions for the Streamlit user. The app asks for these features by hand, so every dropped feature is a dropdown nobody has to fill in. A 30-field form gets abandoned; a 16-field one is merely tedious. This is the real return on the exercise and it is a deployment argument rather than a metric one.

**This is feature selection, not feature engineering, stated explicitly**

The dataset's authors already extracted the 30 columns from raw URLs and page source. The raw URLs are not in the download, so there is no string left to engineer new features from, and constructing interactions of existing binary codes would add nothing the tree models cannot already represent. The spec allows "selection and/or engineering", so selection alone complies, but it deserves saying out loud rather than being left to inference.

**How k was chosen, and the discipline behind it**

Two rankings, chosen because they fail in different ways. `RandomForest` impurity importance is multivariate and judges a feature conditional on what the other features already explain, while mutual information is univariate and judges each feature alone. Averaging their ranks and checking agreement (Spearman ρ = 0.899) works as a cross-check: if two methods with different blind spots order the features the same way, the ordering is a property of the data rather than of either method.

The rankings order the features but say nothing about where to cut. The cut comes from a one-standard-error rule on 5-fold cross-validated F1 inside the training split: take the smallest k whose CV score falls within one SE of the best. Best is 0.9452 at k=30 with SE 0.0042, giving a threshold of 0.9410 and k = 16.

Why the CV had to sit inside the training split. Re-scoring the test set at each k and taking the best would pick whichever cut flatters the test set, and the before and after table would then compare a test-tuned "after" against an untuned "before", a comparison guaranteed to favour selection. The same discipline applies in §5.2 for hyperparameters and §5.3 for the threshold: the test set gets scored once, at the end, for reporting only.

**Cutting on F1 rather than recall, a choice with a visible consequence**

On recall alone the same rule would have kept k = 5, with CV recall 0.9354 at k=5 against 0.9395 at k=30. Features 6 through 16 buy precision, not recall. This is a genuine fork rather than a technicality: I am spending 11 extra dropdowns on precision.

The justification comes from §4.1 and §5. Recall is gameable, since the dummy scores 1.000 on it, so a feature set sized to maximise recall is sized to approach a number a useless model already wins. Precision is no luxury here either, because §1's cost argument depends on false alarms staying rare enough that users keep reading the warnings. A five-feature model that warns constantly destroys its own recall in practice, since people stop looking.

**The §2 prediction survived two model-based methods**

`Favicon` (mean rank 24.5), `popUpWidnow` (24.0) and `Iframe` (29.0) all landed in the bottom seven of 30. That prediction was recorded in §2.3.1.2 from conditional phishing rates alone, before any model was fitted. The ordering of those two cells is the evidence, and it shows the flat-spread screen in §2 was measuring something real rather than serving as a plot exercise.

Where the methods disagreed is more informative than where they agreed. `Submitting_to_email` (RF 21st, MI 30th), `Shortining_Service` (24th against 15th) and `Abnormal_URL` (23rd against 14th) split by around 9 ranks each. The pattern matches the methods' definitions: MI sees a feature's standalone association, while RF sees whether it still earns a split once stronger features have conditioned the node, so a feature carrying information another feature already supplies ranks lower under RF. All three sit below the cut either way, so nothing in the final model rides on it, and I should say that rather than over-claim a finding.

**Two honesty caveats on k = 16**

The cut is flat rather than knife-edge. k = 13 scores 0.9409 against a 0.9410 threshold, and the CV curve stays level from roughly k = 13 through k = 30. Any k in the teens is defensible. The rule picked 16; calling 16 optimal would be false precision.

It is also model-specific. The same rule returns k = 8 for `LogisticRegression` and k = 21 for `RandomForest`. The 16 is tuned to the model §4.2 carried forward. Had that argument landed on a different model, `SELECTED_MODEL` above changes and this entire table gets re-run.

### 5.2 Hyperparameter tuning

`RandomizedSearchCV` only, **exactly 2 hyperparameters, at most 3 values each** — a hard spec constraint. `random_state=RANDOM_STATE` throughout for reproducibility.

That cap makes *which* two the real decision, so it was measured rather than guessed. Four candidates were scanned one-at-a-time on the same training folds, every other parameter left at its default, and scored by cross-validated F1 on the phishing class:

| Hyperparameter | Values scanned | CV F1 across them | Spread |
| --- | --- | --- | --- |
| `learning_rate` | 0.05 / 0.1 / 0.2 | 0.9333 → 0.9426 → 0.9459 | **0.0126** |
| `max_depth` | 2 / 3 / 5 | 0.9351 → 0.9426 → 0.9466 | **0.0115** |
| `n_estimators` | 100 / 200 / 300 | 0.9426 → 0.9447 → 0.9460 | 0.0034 |
| `subsample` | 0.8 / 1.0 | 0.9405 → 0.9426 | 0.0021 |

`learning_rate` and `max_depth` move the score three to six times as much as the alternatives, and searched together they reach 0.9491 — better than any other pair of these four. They are also the two that are not redundant with each other: `max_depth` sets how much each tree can express, `learning_rate` how much of it is kept.

**The grid boundary was checked.** The best point (`max_depth=5`, `learning_rate=0.2`) sits at the top corner of a 3×3 grid, which usually means the grid was drawn too small. Extending the scan to `max_depth` ∈ {3, 5, 8, 12} × `learning_rate` ∈ {0.1, 0.2, 0.3, 0.5} makes it *worse* in every direction — 0.9430 at `lr=0.3`, 0.9375 at `depth=8`, 0.9339 at `depth=12`, `lr=0.2`. So this is a genuine interior peak that the 3-value cap happens to place at a corner, not a wall the search ran into.


In [ ]:
## RandomizedSearchCV
# Exactly two hyperparameters, three values each - the spec's cap. The markdown above
# records why these two and not the other candidates.
#
# Same `cv` folds and same `scoring` as the 5.1 sweep, deliberately: the untuned number
# this is compared against was produced on those folds, so reusing them keeps the
# before/after difference attributable to the parameters rather than to the resampling.

param_distributions = {
    'max_depth': [2, 3, 5],
    'learning_rate': [0.05, 0.1, 0.2],
}

# 3 x 3 = 9 candidates, and n_iter=9 samples all of them without replacement. At this
# size a smaller n_iter would throw away candidates for nothing - that is a consequence
# of the <=3-values-per-parameter cap, not an oversight. refit='f1' for the same reason
# the 5.1 cut was made on F1: recall alone is gameable.
search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_distributions=param_distributions,
    n_iter=9, cv=cv, scoring=scoring, refit='f1',
    random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True,
).fit(X_train[SELECTED_FEATURES], y_train)

cv_results = pd.DataFrame(search.cv_results_)
cv_results.to_csv(ARTIFACTS / 'search_results.csv', index=False)

print(f'best params : {search.best_params_}')
print(f'best CV F1  : {search.best_score_:.4f}')
print(f'defaults    : max_depth=3, learning_rate=0.1 -> CV F1 '
      f'{cv_results.loc[(cv_results.param_max_depth == 3) & (cv_results.param_learning_rate == 0.1), "mean_test_f1"].iloc[0]:.4f}')
print()
print(cv_results[['param_max_depth', 'param_learning_rate', 'mean_test_f1', 'std_test_f1',
                  'mean_test_recall', 'mean_train_f1', 'rank_test_f1']]
      .sort_values('rank_test_f1').round(4).to_string(index=False))


In [ ]:
## The 3 x 3 surface, so the best point can be seen in context
# A single "best params" line hides whether the winner is a peak or a coin flip between
# neighbours. Two panels: what cross-validation scored, and how far each candidate's
# training score ran ahead of it (the overfitting the deeper trees are buying).

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (col, title, cmap) in zip(axes, [
        ('mean_test_f1', 'Cross-validated F1 (phishing)', 'viridis'),
        ('gap', 'Train F1 minus CV F1 (overfitting)', 'rocket_r')]):
    grid = cv_results.assign(gap=cv_results.mean_train_f1 - cv_results.mean_test_f1) \
        .pivot(index='param_max_depth', columns='param_learning_rate', values=col)
    sns.heatmap(grid.astype(float), annot=True, fmt='.4f', cmap=cmap, ax=ax,
                cbar=False, linewidths=0.5)
    ax.set(title=title, xlabel='learning_rate', ylabel='max_depth')
fig.tight_layout()
plt.show()


In [ ]:
## Tuned vs untuned on the test set
# The untuned row is read back from 5.1's saved metrics rather than refitted: same 16
# features, same split, same fit-once-score-once protocol, so the only thing that
# differs between the two rows is the two hyperparameters.

untuned = pd.read_csv(ARTIFACTS / 'reduced_metrics.csv')
untuned = untuned[untuned.model == SELECTED_MODEL].iloc[0]

tuned_model = search.best_estimator_  # refit on the full training split by refit='f1'
pred_te = tuned_model.predict(X_test[SELECTED_FEATURES])
pred_tr = tuned_model.predict(X_train[SELECTED_FEATURES])

tuned = {
    'test_recall': recall_score(y_test, pred_te, pos_label=-1, zero_division=0),
    'test_precision': precision_score(y_test, pred_te, pos_label=-1, zero_division=0),
    'test_f1': f1_score(y_test, pred_te, pos_label=-1, zero_division=0),
    'test_accuracy': accuracy_score(y_test, pred_te),
    'train_test_gap': accuracy_score(y_train, pred_tr) - accuracy_score(y_test, pred_te),
}

tuning_comparison = pd.DataFrame({
    f'untuned (depth 3, lr 0.1)': {m: untuned[m] for m in tuned},
    f'tuned ({search.best_params_["max_depth"]}, {search.best_params_["learning_rate"]})': tuned,
})
tuning_comparison['change'] = (tuning_comparison.iloc[:, 1] - tuning_comparison.iloc[:, 0])

print(f'{SELECTED_MODEL} on the {K} selected features, deduplicated test set:')
print(tuning_comparison.round(4).to_string())


Test set, untuned vs tuned:

| Metric | Untuned (depth 3, lr 0.1) | Tuned (depth 5, lr 0.2) | Change |
| --- | --- | --- | --- |
| Recall (phishing) | 0.9387 | **0.9421** | +0.0033 |
| Precision | 0.9562 | 0.9421 | −0.0141 |
| F1 | 0.9474 | 0.9421 | −0.0053 |
| Accuracy | 0.9462 | 0.9402 | −0.0060 |
| Train−test gap | 0.0013 | **0.0346** | +0.0333 |

**What tuning did: improved cross-validation, regressed on test, and revealed why.**

`RandomizedSearchCV` settled on `max_depth=5` and `learning_rate=0.2`, taking CV F1 from 0.9426 to 0.9491, a gain of 0.65pp. On the held-out test set F1 went the other way, 0.9474 → 0.9421, along with accuracy (down 0.60pp) and precision (down 1.41pp).

The defensible claim is that tuning produced no real improvement, not that it made things worse. The CV gain of 0.0065 sits against a standard error of 0.0019, and the test losses are around 1pp on a 1,170-row test set where each rate carries roughly ±1pp of sampling noise. Both movements amount to about one standard error, so reading either as a result would be over-reading.

What tuning actually bought shows up in the training score, and it is not performance. Mean train F1 went 0.9489 → 0.9786 and the train to test gap went 0.0013 → 0.0346, a factor of 27. A deeper tree taking larger steps memorises more of the training split. This is the same failure mode `RandomForest` showed in §4.2 and the second time in the project that a higher score proved to be memorisation rather than learning. The recurrence is the finding worth reporting: on this dataset the thing that moves scores upward is almost always capacity, and capacity has to be checked against the gap before anyone believes it.

**Why these two hyperparameters**

The spec caps the search at 2 parameters × 3 values, so which two is the real decision and it should not be made out of habit. I scanned candidates one at a time on the same CV folds with everything else left at default, measuring how far each could move CV F1:

| Parameter | CV F1 spread across its range |
| --- | --- |
| `learning_rate` | 0.0126 |
| `max_depth` | 0.0115 |
| `n_estimators` | 0.0034 |
| `subsample` | 0.0021 |

`learning_rate` and `max_depth` carry roughly four times the leverage of the alternatives, and searched as a pair they reach 0.9491, beating every other pairing. A grid this small only pays off when it is spent on the parameters that can actually move the score.

The boundary was checked as well, so the optimum is not a grid wall. Extending to depth {3,5,8,12} × lr {0.1,0.2,0.3,0.5} scores worse everywhere outside the original region, reaching 0.9375 at depth 8 and 0.9339 at depth 12 with lr 0.2, so (5, 0.2) is an interior peak rather than the edge of a box I drew too small. That scan is deliberately absent from the notebook, since it would read as a four-parameter search and breach the spec's cap, but the numbers are recorded and I can produce them on request.

**Keeping the tuned model, and why the test table is not a reason to switch**

The tuned settings are what a pre-registered procedure returned from cross-validation on the training split. Switching back to the untuned model after seeing the test numbers would mean selecting on the test set, precisely the error §5.1 took care to avoid when choosing k, and it would invalidate the test set as an estimate for everything downstream. Were I to swap, the justification would have to be the train to test gap, which is visible without looking at test metrics at all.

**The one metric that improved is the one §1 cares about, and it still is not the lever**

Recall rose 0.0033 while precision fell 0.0141, which works out at roughly two more phishing sites caught for eight more false alarms. That trade was incidental rather than chosen, since hyperparameters move both metrics at once and in directions I cannot predict beforehand. The decision threshold in §5.3 moves them against each other deliberately and monotonically, which is what a recall-prioritised brief actually needs. That is why the threshold, and not this search, is where §1's cost argument gets applied.

The grid was not the limiting factor. The ceiling comes from the 64 contradictory feature vectors identified in §3.1, rows identical under these 30 features but carrying opposite labels. No hyperparameter setting can separate those, so more searching would not have helped.

### 5.3 Final model

Two things are still undecided at this point, and only one of them is a model choice.

The **estimator** is settled: `GradientBoostingClassifier(max_depth=5, learning_rate=0.2)` on the 16 features from §5.1 — what the pre-registered search returned in §5.2.

The **decision threshold** is not. `predict` hard-codes 0.5, which is the right cut only if a false negative and a false positive cost the same. §1 argues they do not: a missed phishing site puts a user in front of a credential-harvesting form, a false alarm costs one dismissed warning. That asymmetry belongs in the threshold, not in the loss function — §5.2 showed that hyperparameters move precision and recall together and unpredictably, while the threshold moves them against each other on purpose.

So the threshold is chosen the same way `k` was in §5.1: on the **training split only**, from out-of-fold cross-validated probabilities, by a rule fixed in advance — minimise expected cost, where one false negative is worth `FN_TO_FP_COST` false positives. The test set sees the result once, at the end.


In [ ]:
## Choose the decision threshold on out-of-fold training probabilities
# cross_val_predict gives every training row a probability from a model that never saw
# it. Sweeping the threshold on those is honest; sweeping it on the test set would be
# choosing the operating point on the data used to report it.

final_model = GradientBoostingClassifier(random_state=RANDOM_STATE, **search.best_params_)

oof_proba = cross_val_predict(final_model, X_train[SELECTED_FEATURES], y_train,
                              cv=cv, method='predict_proba')
PHISH_COL = list(np.unique(y_train)).index(-1)  # column order follows sorted classes
oof_phishing_proba = oof_proba[:, PHISH_COL]

thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
rows = []
for t in thresholds:
    pred = np.where(oof_phishing_proba >= t, -1, 1)
    tn, fp, fn, tp = confusion_matrix(y_train, pred, labels=[1, -1]).ravel()
    rows.append({'threshold': t, 'false_negatives': fn, 'false_positives': fp,
                 'recall': recall_score(y_train, pred, pos_label=-1, zero_division=0),
                 'precision': precision_score(y_train, pred, pos_label=-1, zero_division=0),
                 'f1': f1_score(y_train, pred, pos_label=-1, zero_division=0)})
threshold_curve = pd.DataFrame(rows)
threshold_curve.to_csv(ARTIFACTS / 'threshold_curve.csv', index=False)

# How sensitive is the answer to the cost ratio? Printed before picking one, so the
# choice is made with the alternatives visible.
print('cost-optimal threshold at several false-negative : false-positive cost ratios')
print('(out-of-fold on the training split, 4,679 rows)\n')
for w in [1, 2, 3, 5, 10, 20]:
    best = threshold_curve.loc[(w * threshold_curve.false_negatives
                                + threshold_curve.false_positives).idxmin()]
    print(f'  1 FN = {w:2d} FP  ->  threshold {best.threshold:.2f}   '
          f'recall {best.recall:.4f}   precision {best.precision:.4f}   '
          f'f1 {best.f1:.4f}   ({int(best.false_negatives)} missed, '
          f'{int(best.false_positives)} false alarms)')

# One number, and section 1 owns it. Everything downstream - the threshold, the final
# confusion matrix, what the Streamlit app warns on - follows from this line.
FN_TO_FP_COST = 5

THRESHOLD_ = threshold_curve.loc[(FN_TO_FP_COST * threshold_curve.false_negatives
                                  + threshold_curve.false_positives).idxmin()]
DECISION_THRESHOLD = float(THRESHOLD_.threshold)
print(f'\nchosen: 1 FN = {FN_TO_FP_COST} FP  ->  threshold {DECISION_THRESHOLD:.2f}')
print(f'for comparison, the F1-maximising threshold is '
      f'{threshold_curve.loc[threshold_curve.f1.idxmax(), "threshold"]:.2f} - '
      f'near 0.50, which is what "no cost asymmetry" would have given.')


In [ ]:
## The curve behind that choice
# Left: precision and recall crossing as the threshold moves - the trade the cost ratio
# is resolving. Right: expected cost at each ratio, so the flatness of the minimum is
# visible rather than implied by a single chosen number.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(threshold_curve.threshold, threshold_curve.recall, label='recall (phishing)')
axes[0].plot(threshold_curve.threshold, threshold_curve.precision, label='precision')
axes[0].plot(threshold_curve.threshold, threshold_curve.f1, label='F1', ls='--', c='grey')
axes[0].axvline(DECISION_THRESHOLD, c='crimson', ls='--',
                label=f'chosen t = {DECISION_THRESHOLD:.2f}')
axes[0].axvline(0.5, c='black', ls=':', lw=1, label="sklearn's default 0.50")
axes[0].set(xlabel='threshold on P(phishing)', ylabel='out-of-fold score',
            title='Precision and recall trade against each other')
axes[0].legend(fontsize=8)

for w in [1, 2, 3, 5, 10]:
    cost = (w * threshold_curve.false_negatives + threshold_curve.false_positives)
    axes[1].plot(threshold_curve.threshold, cost / cost.min(),
                 label=f'1 FN = {w} FP', lw=1.6 if w == FN_TO_FP_COST else 1,
                 alpha=1.0 if w == FN_TO_FP_COST else 0.5)
axes[1].axvline(DECISION_THRESHOLD, c='crimson', ls='--')
axes[1].set(xlabel='threshold on P(phishing)', ylabel='expected cost / its own minimum',
            title='Where the cost minimum sits depends on the ratio', ylim=(0.95, 2.0))
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()


In [ ]:
## Final evaluation: the test set, scored once
# Fit on the full training split, predict the test split once, at both thresholds. The
# 0.50 column is not a second candidate - it is there so the cost of the chosen
# operating point is visible instead of asserted.

final_model.fit(X_train[SELECTED_FEATURES], y_train)
test_phishing_proba = final_model.predict_proba(X_test[SELECTED_FEATURES])[:, PHISH_COL]
pred_default = np.where(test_phishing_proba >= 0.5, -1, 1)
pred_final = np.where(test_phishing_proba >= DECISION_THRESHOLD, -1, 1)

summary = {}
for label, pred in [('threshold 0.50', pred_default),
                    (f'threshold {DECISION_THRESHOLD:.2f} (chosen)', pred_final)]:
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[1, -1]).ravel()
    summary[label] = {
        'recall (phishing)': recall_score(y_test, pred, pos_label=-1, zero_division=0),
        'precision': precision_score(y_test, pred, pos_label=-1, zero_division=0),
        'f1': f1_score(y_test, pred, pos_label=-1, zero_division=0),
        'accuracy': accuracy_score(y_test, pred),
        'phishing sites missed': fn,
        'false alarms': fp,
    }
final_summary = pd.DataFrame(summary)
final_summary['change'] = final_summary.iloc[:, 1] - final_summary.iloc[:, 0]
final_summary.to_csv(ARTIFACTS / 'final_metrics.csv')
print(final_summary.round(4).to_string())

print(f'\n\nfinal model - {len(SELECTED_FEATURES)} features, '
      f'{search.best_params_}, threshold {DECISION_THRESHOLD:.2f}\n')
print(classification_report(y_test, pred_final, labels=[-1, 1],
                            target_names=['phishing (-1)', 'legitimate (1)'], digits=4))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (label, pred) in zip(axes, [('threshold 0.50', pred_default),
                                    (f'threshold {DECISION_THRESHOLD:.2f} (chosen)', pred_final)]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=[-1, 1], display_labels=['phishing', 'legitimate'],
        cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(label)
fig.tight_layout()
plt.show()


In [ ]:
## Persist for the Streamlit app
# One artefact holding everything the app needs to reproduce this exact decision: the
# fitted estimator, the feature list *in the column order it was trained on*, and the
# threshold. Saving the model alone would leave the app to guess the other two, and a
# column-order mismatch is silent - no error, just wrong answers.

joblib.dump({
    'model': final_model,
    'features': SELECTED_FEATURES,
    'threshold': DECISION_THRESHOLD,
    'phishing_proba_column': PHISH_COL,
    'labels': {-1: 'phishing', 1: 'legitimate'},
    'metadata': {
        'trained_on': 'UCI Phishing Websites (id=327), deduplicated, stratified 80/20',
        'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
        'params': search.best_params_,
        'random_state': RANDOM_STATE,
        'fn_to_fp_cost': FN_TO_FP_COST,
        'test_recall': float(final_summary.iloc[0, 1]),
        'test_precision': float(final_summary.iloc[1, 1]),
    },
}, ARTIFACTS / 'final_model.pkl')

print(f'wrote {ARTIFACTS / "final_model.pkl"}')
print(f'app collects {len(SELECTED_FEATURES)} inputs, in this order:')
for i, f in enumerate(SELECTED_FEATURES, 1):
    print(f'  {i:2d}. {f}')


In [ ]:
## New data
# Two rows built from the training data rather than by hand: for each selected feature,
# the level with the highest and the lowest phishing rate on the *training* split. That
# makes the smoke test a real check of direction - if the "worst case" row does not come
# out phishing, something is wired backwards - instead of a guess at the encoding.

rates = pd.DataFrame({f: y_train.groupby(X_train[f]).apply(lambda s: (s == -1).mean())
                      for f in SELECTED_FEATURES})

worst_case = rates.idxmax().astype(int)   # most phishing-like level of every feature
best_case = rates.idxmin().astype(int)    # most legitimate-like level of every feature
new_data = pd.DataFrame([worst_case, best_case], index=['phishing-looking site',
                                                        'legitimate-looking site'])
print(new_data.to_string())

## Predict
proba = final_model.predict_proba(new_data[SELECTED_FEATURES])[:, PHISH_COL]
verdict = np.where(proba >= DECISION_THRESHOLD, 'PHISHING', 'legitimate')

print(f'\nthreshold {DECISION_THRESHOLD:.2f}\n')
for name, p, v in zip(new_data.index, proba, verdict):
    print(f'  {name:24s}  P(phishing) = {p:.4f}  ->  {v}')

# A middling row too: the app has to behave sensibly between the extremes, not only on
# them. Half the features set to their phishing level, half to their legitimate level.
mixed = pd.DataFrame([{f: (worst_case[f] if i % 2 == 0 else best_case[f])
                       for i, f in enumerate(SELECTED_FEATURES)}]).astype(int)
p_mixed = final_model.predict_proba(mixed[SELECTED_FEATURES])[:, PHISH_COL][0]
print(f'  {"half-and-half site":24s}  P(phishing) = {p_mixed:.4f}  ->  '
      f'{"PHISHING" if p_mixed >= DECISION_THRESHOLD else "legitimate"}')


**The operating point**

`GradientBoosting` (`max_depth=5`, `learning_rate=0.2`, `random_state=42`) on the 16 selected features, with the decision threshold moved from 0.50 to 0.08. On the held-out test set of 604 phishing and 566 legitimate sites: recall 0.9901, precision 0.8592, accuracy 0.9111. That is 6 of 604 phishing sites missed at the price of 98 false alarms.

**The assumption it rests on, because it is the weakest link**

The threshold is not learned from the data. It follows from `FN_TO_FP_COST = 5`, a judgement made in §1 that one missed phishing site is worth five false alarms. Nothing in the dataset supports that number, and I would rather say so before being asked than after. What it controls:

| Assumed cost ratio | Threshold | Comment |
| --- | --- | --- |
| 1:1 | 0.51 | "no asymmetry" reproduces sklearn's default cut-off almost exactly |
| 3:1 | 0.27 | |
| 5:1 | 0.08 | the operating point chosen here |
| 10:1 | 0.05 | precision falls further |

The 1:1 case landing on 0.51 is worth noting. The default threshold everyone uses silently assumes the two errors cost the same, which for this application is a substantive claim and a wrong one. Choosing 0.08 is therefore not a departure from a neutral setting; it replaces an unexamined assumption with a stated one.

As with k in §5.1 and the hyperparameters in §5.2, the threshold was chosen on out-of-fold training probabilities (`cross_val_predict`) and applied to the test set exactly once, for reporting.

**What the operating point costs, reported honestly**

Recall 0.9901 is the headline and on its own it misleads. Alongside it:

Precision fell to 0.8592, so roughly one warning in seven is a false alarm. §1's alert-fatigue argument identifies this as the number to watch if the ratio gets pushed higher.

Accuracy fell as well, 0.9402 → 0.9111. A more accurate model was available and I chose deliberately not to deploy it, for the reason given in §1.

This is why the threshold-0.50 column stays in the table. A reader shown only the 0.9901 is being misled about what got traded away.

**The error floor is in the data, not the model**

§3.1 found 64 feature vectors appearing under both labels, pairs of real websites identical under these 30 features but labelled differently. No classifier reading only these features can separate them, so a perfect score was never on offer. Worth stating before someone asks why the number is not 99.9%.

**Two encodings run backwards from the documentation**

Both surfaced through the smoke-test row above, which is built from measured per-level phishing rates rather than from what the codes are supposed to mean, and that is the only reason either came to light.

`Domain_registeration_length` codes 1 as "registered for more than a year", nominally the legitimate level, yet that level runs 60.2% phishing against 36.4% for the short-registration level.

`SSLfinal_State` is non-monotone: the "suspicious" middle level 0 runs 98.2% phishing, worse than the "no HTTPS" level at 85.8%.

Both hold on the full 11,055 rows, so neither is a deduplication artefact. This is why the model treats a long-lived domain as mildly suspicious, and it retrospectively justifies §2's decision to screen features on conditional phishing rate per level rather than trusting the documented direction of each code.

**Deployment limits**

The model reads 16 pre-extracted features rather than a URL. The Streamlit app asks the user to supply them by hand, which is honest for a coursework demonstration but is not a product. A real deployment needs the extraction pipeline that produced this dataset, including the third-party lookups (`web_traffic`, `Page_Rank`, `Google_Index`) that §1 flagged as an availability assumption.

The data is a 2015 snapshot and phishing kits move on. §2.3.1.2 already shows three features that have gone dead since collection. Others will follow, and the model has no way of noticing.

Every number here comes from a single 80/20 split of 5,849 deduplicated rows. A 1,170-row test set carries roughly ±1pp of sampling noise on each rate, so differences below about a point, which includes most of §5.2's, should not be treated as real. Reporting four decimals throughout is a convention for comparing runs rather than a claim that all four carry meaning.

The adversary is not static. Several of the 16 features are cheap for an attacker to flip once they know they are measured. A static classifier trained once assumes an adversary who does not adapt, which is the least true assumption in the whole project.